**[Course Home](../README.md) | [Python Lab](../python/04_mutual_information_and_venn_geometry.ipynb) | [Prev: Ch 03: Where Fleet Priors Come From](03_maximum_entropy_priors.ipynb) | [Next: Ch 05: The Geometry of Inference](05_curvature_laplace_and_fisher_information.ipynb)**

*Part II: Multi-Variable Geometry & Causal Architectures — Chapter 04 of 12*

---

# Chapter 04: Shared Outages & Venn Geometry
### *Joint Entropy, Mutual Information, Venn Geometry, & Reversal Paradox*

---

## 1. The Scenario: The 2:00 PM Double Failure

At 2:15 PM, two completely unrelated unit tests fail simultaneously on your CI
dashboard:
1. `test_user_authentication` ($T_1 = 1$)
2. `test_billing_checkout` ($T_2 = 1$)

A junior engineer panics:
> *"The authentication service and the billing service are both broken! Did
> someone push a bug to core business logic?"*

A senior site reliability engineer (SRE) checks the database dashboard and
responds:
> *"Wait. Both of those tests connect to the staging Redis cache ($Z$). At 2:14
> PM, Redis suffered an eviction spike. The tests didn't break each other—Redis
> failed them both."*

We formalize this operational diagnosis as a **3-node Bayesian causal network**
where a hidden infrastructure state $Z \in \{0, 1\}$ acts as a common cause
(confounder) for tests $T_1$ and $T_2$:

![The Causal Fork Architecture: Common Cause and Information Flow](../images/ch04_causal_fork.png)

In this causal graph, each component has a precise physical meaning:
* **Nodes represent system variables**: The root node $Z$ represents the
  physical operating state of the shared Redis cluster ($0 = \text{healthy}$,
  $1 = \text{degraded}$). The child nodes $T_1$ and $T_2$ represent observable
  binary test outcomes on the CI dashboard ($0 = \text{pass}$,
  $1 = \text{fail}$).
* **Directed arrows represent physical cause-and-effect**: Arrows point from
  $Z$ into $T_1$ and $T_2$ because database degradation physically causes test
  assertions to time out and fail.
* **$Z$ is "hidden" (latent)**: Continuous integration dashboards and automated
  build runners only record whether individual tests passed or failed; they do
  not directly monitor the internal memory eviction rate or socket queue depth
  of the Redis cluster. We must infer the hidden state $Z$ purely from the
  pattern of observed test failures.
* **$Z$ is a confounder (common cause)**: The authentication service does not
  call or depend on the billing service, and an authentication test failure
  cannot physically trigger a billing test failure. Instead, the shared Redis
  outage breaks both services simultaneously. This common cause creates an
  apparent, purely observational correlation between $T_1$ and $T_2$ that
  vanishes the moment the state of the shared infrastructure is known.

The physical infrastructure parameters are:

$$
P(Z = 1) = 0.02 \quad (\text{2% chance shared database is degraded})
$$

- When infra is healthy ($Z = 0$): tests fail independently at baseline flake
  rate $P(T_i = 1 \mid Z = 0) = 0.01$.
- When infra is degraded ($Z = 1$): both tests fail at high rate
  $P(T_i = 1 \mid Z = 1) = 0.80$.

---

### Step 1: Bayesian Common-Cause Diagnosis

When both $T_1 = 1$ and $T_2 = 1$ fail on the same build, what is the posterior
probability that our shared infrastructure suffered an outage?

We apply Bayes' Theorem. Marginally, $T_1$ and $T_2$ are strongly correlated
because both reflect the health of the underlying database. But once you know
the actual state of Redis ($Z$), Auth and Billing flip completely separate
coins—knowing that Auth failed provides zero extra information about Billing
given $Z$. Under conditional independence given $Z$:

$$
P(T_1 = 1, T_2 = 1 \mid Z = 1) = 0.80 \times 0.80 = 0.6400
$$

$$
P(T_1 = 1, T_2 = 1 \mid Z = 0) = 0.01 \times 0.01 = 0.0001
$$

The marginal probability of observing a simultaneous double failure is:

$$
P(T_1 = 1, T_2 = 1) = P(Z = 1) P(T_1 = 1, T_2 = 1 \mid Z = 1)
  + P(Z = 0) P(T_1 = 1, T_2 = 1 \mid Z = 0)
$$

$$
P(T_1 = 1, T_2 = 1) = 0.02(0.64) + 0.98(0.0001) = 0.0128 + 0.000098 = 0.012898
$$

Now compute the posterior probability of a shared infrastructure outage:

$$
P(Z = 1 \mid T_1 = 1, T_2 = 1) = \frac{0.02 \times 0.64}{0.012898}
  = \frac{0.0128}{0.012898} \approx \mathbf{99.24\%}!
$$

Contrast this with observing only a single test failure $T_1 = 1$:

$$
P(T_1 = 1) = 0.02(0.80) + 0.98(0.01) = 0.0160 + 0.0098 = 0.0258
$$

$$
P(Z = 1 \mid T_1 = 1) = \frac{0.02 \times 0.80}{0.0258}
  = \frac{0.0160}{0.0258} \approx 62.02\%
$$

A single failure leaves substantial ambiguity ($37.98\%$ chance the test was
just an isolated baseline flake). But a simultaneous double failure escalates
our outage certainty to **$99.24\%$**!

---

### Step 2: The Information-Theoretic Link

How does this Bayesian causal structure map to Information Theory?

1. **Marginal Mutual Information is Positive ($I(T_1; T_2) > 0$):**
   Marginally (averaging over $Z$), knowing that $T_1 = 1$ provides information
   about $T_2$:

$$
P(T_2 = 1 \mid T_1 = 1) = \frac{P(T_1 = 1, T_2 = 1)}{P(T_1 = 1)}
  = \frac{0.012898}{0.0258} \approx 0.4999 \gg 0.0258
$$

   Because both tests carry information about the hidden common cause $Z$, they
   share positive Mutual Information ($0.0481\text{ bits}$ of shared overlap,
   which we preview here and define formally in Section 3 below):

$$
I(T_1; T_2) \approx 0.0481\text{ bits}
$$

2. **Conditional Mutual Information Vanishes ($I(T_1; T_2 \mid Z) = 0$):**
   Once an SRE checks the Redis metrics and confirms $Z = 1$ (or $Z = 0$), the
   tests become strictly conditionally independent:

$$
P(T_1, T_2 \mid Z) = P(T_1 \mid Z) P(T_2 \mid Z)
$$

   The conditional mutual information collapses to exactly zero:

$$
I(T_1; T_2 \mid Z) = 0.0000\text{ bits}
$$

   Conditioning on the common cause **destroys the apparent correlation**!

---

## 2. The Linear Blind Spot of Classical Correlation

For more than a century, classical statistics measured the relationship between
two variables using the **Pearson correlation coefficient** $r \in [-1, 1]$:

$$
r = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y}
$$

Pearson correlation has a fatal limitation: **it measures exclusively linear
(straight-line) relationships**.

Consider an intermittent hardware race condition where sensor reading $Y$ is a
deterministic quadratic function of CPU load $X \sim \mathcal{U}[-1, 1]$:

$$
Y = X^2
$$

Because $X$ is symmetric around zero:

$$
\mathbb{E}[X] = 0, \quad \mathbb{E}[X^3] = 0 \implies
\text{Cov}(X, Y) = \mathbb{E}[X^3] - \mathbb{E}[X]\mathbb{E}[Y] = 0
$$

$$
\mathbf{r = 0.0000}
$$

A standard linear analysis concludes that $X$ and $Y$ are completely
unrelated! Yet in physical reality, $Y$ is **100% determined by $X$**. If you
know $X = 0.7$, you know $Y = 0.49$ with absolute certainty.

Information Theory provides the universal, non-linear, non-parametric metric
of dependence: **Mutual Information $I(X; Y)$**.

---

## 3. Mutual Information & The Information Venn Diagram

Before analyzing two variables together, we define their **Joint Entropy**
$H(X, Y)$:

$$
\mathbf{H(X, Y) \equiv -\sum_{x, y} P(x, y) \log_2 P(x, y) \quad [\text{bits}]}
$$

Joint entropy represents the **total union** of information in the system—the
number of bits needed to store both outcomes simultaneously.

To compute the joint entropy of our two CI tests, we marginalize over the
hidden Redis state $Z$ to obtain the exact **$2 \times 2$ Marginal Joint
Probability Table** $P(T_1, T_2)$:

$$
\begin{array}{c|cc|c}
P(T_1, T_2) & T_2 = 0 \text{ (Pass)} & T_2 = 1 \text{ (Fail)} & P(T_1) \\
\hline
T_1 = 0 \text{ (Pass)} & 0.9613 & 0.0129 & 0.9742 \\
T_1 = 1 \text{ (Fail)} & 0.0129 & 0.0129 & 0.0258 \\
\hline
P(T_2) & 0.9742 & 0.0258 & 1.0000
\end{array}
$$

Summing $-P(t_1, t_2) \log_2 P(t_1, t_2)$ over all four cells yields:

$$
H(T_1, T_2) = -(0.9613 \log_2 0.9613 + 3 \times 0.0129 \log_2 0.0129)
  = 0.2977\text{ bits}
$$

### The Overlap: Mutual Information $I(X; Y)$

The amount of information shared between $X$ and $Y$ is the **Mutual
Information**:

$$
\mathbf{I(X; Y) \equiv \sum_{x, y} P(x, y) \log_2 \frac{P(x, y)}{P(x)P(y)}}
  \quad [\text{bits}]
$$

This expression connects directly to the foundational principle of Chapter 02:

$$
I(X; Y) \equiv D_{\text{KL}}(P(X, Y) \parallel P(X)P(Y))
$$

Recall from Chapter 02 that Kullback-Leibler divergence measures the exact code
penalty of using an incorrect distribution. Here, $I(X; Y)$ is the KL
divergence from the true joint distribution $P(X, Y)$ to the factored product
distribution $P(X)P(Y)$ that assumes independence. In plain English,
**mutual information is the bit penalty of pretending $X$ and $Y$ are
independent**.

* If $X$ and $Y$ are strictly independent:
  $P(x, y) = P(x)P(y) \implies D_{\text{KL}} = 0 \implies \mathbf{I(X; Y) = 0}$.
* If $X$ and $Y$ are identical twins ($Y = X$):
  $\mathbf{I(X; Y) = H(X)}$. Knowing $X$ completely resolves $Y$.

---

### The Information Venn Partition

In 1991, Raymond Yeung proved that Shannon information measures form a
**signed measure** over the space of random variables. Information obeys the
exact geometry of a Venn diagram:

![The Information Venn Partition: Entropy and Mutual Information Geometry](../images/ch04_info_venn.png)

In classical measure theory (such as area or probability), measures are
strictly non-negative. However, Yeung showed that information measures form
a **signed measure**: while 1-variable entropies and 2-variable mutual
informations are always non-negative, the 3-variable intersection (the
interaction information $I(X; Y; Z)$) can become strictly negative when
variables interact synergistically!

From this geometric duality, every fundamental identity can be read off by eye:
1. **The Whole is the Sum of Disjoint Partitions**:

$$
H(T_1, T_2) = H(T_1 \mid T_2) + I(T_1; T_2) + H(T_2 \mid T_1)
$$

2. **Mutual Information is the Intersection**:

$$
I(T_1; T_2) = H(T_1) + H(T_2) - H(T_1, T_2)
$$

$$
I(T_1; T_2) = H(T_1) - H(T_1 \mid T_2) = H(T_2) - H(T_2 \mid T_1)
$$

3. **Variation of Information is the Symmetric Difference**:

$$
\mathbf{VI(T_1; T_2) \equiv H(T_1, T_2) - I(T_1; T_2)}
  = H(T_1 \mid T_2) + H(T_2 \mid T_1) = 0.2496\text{ bits}
$$

In plain English, **Variation of Information** measures the total unshared bits
between two variables—the uncertainty remaining in $T_1$ given $T_2$ plus the
uncertainty remaining in $T_2$ given $T_1$. Unlike mutual information,
$VI(X; Y)$ is a true mathematical distance metric: it satisfies non-negativity
($VI \ge 0$), symmetry ($VI(X; Y) = VI(Y; X)$), identity of indiscernibles
($VI(X; Y) = 0 \iff X = Y$), and the triangle inequality
$VI(X; Z) \le VI(X; Y) + VI(Y; Z)$.

---

## 4. The Causal Architectures: Confounders vs. Colliders (The Reversal Paradox)

In marginal entropy, **conditioning never increases uncertainty on average**:

$$
H(X \mid Z) \le H(X)
$$

*"Information can never hurt."*

It is tempting to assume by false analogy that conditioning on a third variable
$Z$ should also reduce or preserve mutual information:

$$
I(X; Y \mid Z) \stackrel{?}{\le} I(X; Y) \quad \text{--- DANGEROUS FALLACY!}
$$

**Conditioning on a third variable $Z$ can dramatically increase mutual
information!**

To understand why, we compare the three elemental causal DAG motifs: **The Fork
(Confounder)**, **The Chain (Mediator)**, and **The Collider (Berkson's Bias /
XOR)**.

In causal graphical models, **d-separation** (directional separation) formalizes
the governing rule of information flow: conditioning on an intermediate node
blocks paths through Forks and Chains, but conditioning on a node opens paths
through Colliders.

---

### Architecture A: The Fork / Confounder ($X \leftarrow Z \to Y$)

In our 2:15 PM outage scenario:
* $Z$: Staging Redis cache health ($0 = \text{Healthy}$, $1 = \text{Degraded}$).
* $X = T_1$: Auth test failure.
* $Y = T_2$: Billing test failure.

Because both tests depend on the common cause Redis, they exhibit positive
unconditional mutual information:

$$
I(T_1; T_2) \approx 0.0481\text{ bits} > 0
$$

To a naive observer, auth failures seem to predict billing failures! However,
once an SRE logs into Redis and conditions on $Z$:

$$
I(T_1; T_2 \mid Z) = 0.0000\text{ bits}
$$

**Conditioning on the common cause blocks the fork and destroys the mutual
information.** The correlation was completely spurious, mediated through $Z$.

---

### Architecture B: The Chain / Mediator ($X \to Z \to Y$)

Now consider an end-to-end deployment pipeline:
* $X$: A flawed Git commit containing an invalid SQL migration.
* $Z$: A database schema migration failure on the staging host.
* $Y$: Downstream user API requests timing out with 500 errors.

Here, the bad commit causes the schema migration failure, which in turn causes
the API timeouts ($X \to Z \to Y$).

Unconditionally, Bad Commits and API Timeouts share positive mutual
information: observing a bad commit makes API timeouts much more probable
($I(X; Y) > 0$).

However, if an engineer inspects the database host and directly conditions on
the mediator $Z$ (knowing whether the schema migration succeeded or failed),
knowing whether the commit was buggy gives zero extra information about API
timeouts:

$$
I(X; Y \mid Z) = 0.0000\text{ bits}
$$

**Conditioning on the intermediate mediator blocks the chain**, exactly like
conditioning on the common cause in a fork!

---

### Architecture C: The Collider / Berkson's Bias ($X \to Z \leftarrow Y$)

Now consider two completely independent services:
* $X$: Payment gateway timeout ($50/50$ random flake).
* $Y$: Email worker thread crash ($50/50$ random flake).

Because they are independent:

$$
I(X; Y) = 0.0000\text{ bits}
$$

Now suppose your CI runner has a shared queue limit $Z$: the build fails if
**either $X$ fails OR $Y$ fails (or both)**:

$$
Z = X \text{ OR } Y
$$

Now, condition on observing a build failure ($Z = 1$):
* Suppose you inspect Worker 1 and find that the payment gateway was
  completely fine ($X = 0$).
* You ask: *"Wait, how did the build fail if the payment gateway was healthy?"*
* Because $(0, 0)$ is impossible under $Z = 1$, **the email worker must have
  crashed ($Y = 1$) with 100% certainty!**

$$
P(Y = 1 \mid X = 0, Z = 1) = 1.00 \quad (100\%)
$$

Knowing $X = 0$ completely determines $Y = 1$ (and knowing $X = 1$ cuts the
options), creating mutual information where there was none:

$$
\mathbf{I(X; Y \mid Z = 1) > 0}
$$

> **The Reversal Paradox**: Normally, conditioning shrinks marginal uncertainty
> on average ($H(X \mid Z) \le H(X)$). But on a collider, conditioning **creates
> mutual information out of thin air**: $I(X; Y \mid Z) > I(X; Y) = 0$! In
> statistics and causal inference, this phenomenon is called **"Explaining
> Away"** or **Berkson's Paradox**.

---

## 5. Interaction Information: Synergy vs. Redundancy

To formalize this mathematically, we define **3-Way Interaction Information**:

$$
\mathbf{I(X; Y; Z) \equiv I(X; Y) - I(X; Y \mid Z)}
$$

* **Redundancy ($I(X; Y; Z) > 0$ / The Fork & Chain)**:
  Conditioning on $Z$ reduces mutual information ($I(X; Y \mid Z) < I(X; Y)$).
  The variables share duplicate information about $Z$; learning one diminishes
  the incremental value of learning the other. In our Redis scenario:

$$
I(T_1; T_2; Z) = 0.0481 - 0.0000 = +0.0481\text{ bits} > 0
$$

* **Synergy ($I(X; Y; Z) < 0$ / The Collider)**:
  Conditioning on $Z$ increases mutual information ($I(X; Y \mid Z) > I(X; Y)$).
  The variables hold separate puzzle pieces that only reveal information when
  combined together; conditioning on the common outcome exposes interactions
  that did not exist in isolation.
  In the OR alert scenario, $P(Z = 1) = 0.75$, and given $Z = 1$ the three
  outcomes $(0, 1), (1, 0), (1, 1)$ have equal probability $1/3$:

$$
I(X; Y \mid Z = 1) = 2 H_2(1/3) - \log_2 3
  \approx 2(0.9183) - 1.5850 = 0.2516\text{ bits}
$$

$$
I(X; Y \mid Z) = 0.75 \times 0.2516 = 0.1887\text{ bits}
$$

$$
I(X; Y; Z) = 0.0000 - 0.1887 = -0.1887\text{ bits} < 0
$$

---

## 6. Nuisance Information: The Secret to Machine Learning Generalization

This brings us to a fundamental insight in modern deep learning and systems
diagnosis.

Suppose an autonomous triage bot ingests build logs $L$ to classify the root
cause of an outage $Y$. This process forms a Markov chain
$Y \to \text{Logs} \to R$, where the true defect $Y$ generates system logs, and
the bot extracts an internal representation $R$. The total mutual information
between the raw logs and the internal representation $R$ decomposes as:

$$
\mathbf{I(\text{Logs}; R) = \underbrace{I(Y; R)}_{\text{Signal}}
  + \underbrace{I(\text{Logs}; R \mid Y)}_{\text{Nuisance Information}}}
$$

* **$I(Y; R)$**: The true causal defect signal (e.g., database deadlock).
* **$I(\text{Logs}; R \mid Y)$**: Nuisance variation—worker hostnames,
  microsecond timestamps, container IP addresses, ambient CPU loads.

A naive neural network memorizes hostnames and timestamps (overfitting). A
robust, generalizable triage bot acts as a **nuisance information filter**: it
ruthlessly drives $I(\text{Logs}; R \mid Y) \to 0$, compressing raw log noise
onto a clean, invariant manifold where root causes can be classified
effortlessly.

---

## 7. Summary: The Golden Rules of Chapter 04

1. **Mutual Information captures all dependencies**: Unlike Pearson correlation
   $r$, $I(X; Y)$ detects quadratic, non-linear, and logical relationships.
2. **The Information Venn Diagram maps to set theory**:
   * Total system uncertainty = Set Union $H(T_1, T_2) = 0.2977\text{ bits}$
   * Shared diagnostic information:
     Set Intersection $I(T_1; T_2) = 0.0481\text{ bits}$
   * Residual doubt = Set Difference $H(T_1 \mid T_2) = 0.1248\text{ bits}$
   * Variation of Information:
     Symmetric Difference $VI(T_1; T_2) = 0.2496\text{ bits}$
3. **The Fork, Chain, and Collider Rules**:
   * Conditioning on a common cause (Fork) **destroys** mutual information
     ($I(T_1; T_2 \mid Z) = 0$).
   * Conditioning on an intermediate mediator (Chain) **blocks**
     information flow ($I(X; Y \mid Z) = 0$).
   * Conditioning on a common effect (Collider) **creates** mutual information
     ($I(X; Y \mid Z) > I(X; Y)$).
   * **d-Separation**: Conditioning blocks paths through Forks and Chains, but
     opens paths through Colliders.
4. **Bayesian Diagnosis Explains the Venn Overlap**: Simultaneous failures
   drastically escalate the posterior probability of a shared infrastructure
   outage ($99.24\%$ vs. $62.02\%$ for a single failure) precisely because the
   tests share mutual information mediated through $Z$.
5. **Negative Interaction Information indicates synergy**: $I(X; Y; Z) < 0$ is
   the mathematical signature of colliders and synergistic interactions.

Now, open the companion Python Lab to simulate the 2:15 PM double failure,
verify the Venn partitions numerically, and witness the Reversal Paradox in
code!

---

**[Course Home](../README.md) | [Python Lab](../python/04_mutual_information_and_venn_geometry.ipynb) | [Prev: Ch 03: Where Fleet Priors Come From](03_maximum_entropy_priors.ipynb) | [Next: Ch 05: The Geometry of Inference](05_curvature_laplace_and_fisher_information.ipynb)**